# Practical 9 (B4) - Data Wrangling
### Data Wrangling on Real Estate Market

**Setup:** Import required libraries.

In [ ]:
import pandas as pd
import numpy as np
import matplotlib.pyplot as plt
import seaborn as sns
from sklearn.preprocessing import LabelEncoder

sns.set(style="whitegrid")
print("Libraries ready")

**Task 1:** Import the "RealEstate_Prices.csv" dataset. Clean column names by removing spaces, special characters, or renaming them for clarity.

In [ ]:
url = "RealEstate_Prices.csv"
data = pd.read_csv(url)

cols = ["Neighborhood", "BldgType", "HouseStyle", "OverallQual", "YearBuilt",
        "YrSold", "LotFrontage", "LotArea", "GrLivArea", "TotalBsmtSF", "SalePrice"]
df = data[cols].copy()

df.columns = df.columns.str.strip().str.lower().str.replace(" ", "_")
df = df.rename(columns={
    "bldgtype": "property_type",
    "housestyle": "house_style",
    "saleprice": "sale_price",
    "yrsold": "year_sold",
    "yearbuilt": "year_built"
})

print("Cleaned column names:", list(df.columns))
df.head()

**Task 2:** Handle missing values in the dataset, deciding on an appropriate strategy (e.g., imputation or removal).

In [ ]:
print("Missing values per column:")
print(df.isnull().sum())

df["lotfrontage"] = df["lotfrontage"].fillna(df["lotfrontage"].median())

print("\nTotal missing after imputation:", df.isnull().sum().sum())

**Task 3:** Perform data merging if additional datasets with relevant information are available (e.g., neighborhood demographics or nearby amenities).

In [ ]:
demographics = pd.DataFrame({"neighborhood": df["neighborhood"].unique()})
np.random.seed(1)
demographics["avg_income"] = np.random.randint(40000, 120000, len(demographics))
demographics["population"] = np.random.randint(2000, 15000, len(demographics))

df = df.merge(demographics, on="neighborhood", how="left")

print("Shape after merging:", df.shape)
df.head()

**Task 4:** Filter and subset the data based on specific criteria, such as a particular time period, property type, or location.

In [ ]:
filtered_df = df[(df["property_type"] == "1Fam") & (df["year_sold"] >= 2009)]

print("Single-family homes sold in 2009 or later:", filtered_df.shape[0], "rows")
filtered_df.head()

**Task 5:** Handle categorical variables by encoding them appropriately (e.g., one-hot encoding or label encoding) for further analysis.

In [ ]:
df_encoded = df.copy()
df_encoded = pd.get_dummies(df_encoded, columns=["property_type", "house_style"])

le = LabelEncoder()
df_encoded["neighborhood_encoded"] = le.fit_transform(df_encoded["neighborhood"])

print("Shape after encoding:", df_encoded.shape)
df_encoded.head()

**Task 6:** Aggregate the data to calculate summary statistics or derived metrics such as average sale prices by neighborhood or property type.

In [ ]:
avg_by_neighborhood = df.groupby("neighborhood")["sale_price"].mean().sort_values(ascending=False)
avg_by_property = df.groupby("property_type")["sale_price"].mean().sort_values(ascending=False)

print("Average sale price by property type:")
print(avg_by_property)

plt.figure(figsize=(10, 4))
avg_by_neighborhood.plot(kind="bar", color="teal")
plt.title("Average Sale Price by Neighborhood")
plt.ylabel("Average Sale Price")
plt.tight_layout(); plt.show()

**Task 7:** Identify and handle outliers or extreme values in the data that may affect the analysis or modeling process.

In [ ]:
num_cols = ["sale_price", "grlivarea", "lotarea"]

for col in num_cols:
    Q1 = df[col].quantile(0.25)
    Q3 = df[col].quantile(0.75)
    IQR = Q3 - Q1
    lower = Q1 - 1.5 * IQR
    upper = Q3 + 1.5 * IQR
    outliers = df[(df[col] < lower) | (df[col] > upper)].shape[0]
    print(f"{col}: {outliers} outliers detected")
    df[col] = df[col].clip(lower, upper)

plt.figure(figsize=(8, 4))
df[num_cols].boxplot()
plt.title("Numeric Columns After Handling Outliers")
plt.tight_layout(); plt.show()

**Export:** Save the wrangled dataset for future analysis.

In [ ]:
df.to_csv("RealEstate_Prices_Wrangled.csv", index=False)
print("Saved as RealEstate_Prices_Wrangled.csv in the notebook folder")